In [ ]:
%%time
# ติดตั้ง libraries ทั้งหมด
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['google-genai', 'docling>=2.31', 'docling-ibm-models>=3.4', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'rank_bm25', 'pymupdf', 'pythainlp', 'scikit-learn', 'rich']:
    _pip_install(_pkg)

# ล้าง cache ของ docling-models เพื่อป้องกัน ONNX file mismatch
import shutil, os
cache_path = os.path.expanduser('~/.cache/huggingface/hub/models--ds4sd--docling-models')
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print('🗑️ ล้าง docling-models cache แล้ว')

print('✅ ติดตั้งเรียบร้อยแล้ว!')
print('⚠️ กรุณา Restart runtime: Runtime → Restart session → แล้ว Run ทุก cell ใหม่')

In [ ]:
%%time
# Import libraries หลัก
import hashlib
import os
import json
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

print('✅ Import สำเร็จ!')

In [ ]:
%%time
# สร้างโฟลเดอร์สำหรับเก็บข้อมูล
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

# ข้อมูลตัวอย่าง: สรุป Case Study ด้าน AI ในประเทศไทย
sample_texts = {

    'case_kmitl.txt': '''กรณีศึกษา: AI ที่สถาบันพระจอมเกล้าเจ้าคุณทหารลาดกระบัง (KMITL)

คณะวิศวกรรมศาสตร์ KMITL พัฒนาระบบ AI สำหรับ Smart Campus
ใช้ IoT sensor รวมกับ Machine Learning วิเคราะห์การใช้พลังงานในอาคารเรียน
ผลการทดสอบพบว่าลดค่าไฟฟ้าได้ 25% และเพิ่มประสิทธิภาพการใช้ห้องเรียน

นอกจากนี้ สาขาวิศวกรรมสารสนเทศ ยังพัฒนาระบบ NLP ภาษาไทย
สำหรับวิเคราะห์ความคิดเห็นนักศึกษาจากแบบประเมิน
ใช้ Sentiment Analysis และ Topic Modeling จัดกลุ่มปัญหา
ช่วยปรับปรุงหลักสูตรได้ตรงจุดมากขึ้น

ล่าสุด KMITL นำ RAG มาสร้างระบบ AI Tutor
ช่วยตอบคำถามวิชาเรียนจากเอกสารประกอบการสอนกว่า 200 วิชา
นักศึกษาสามารถถามคำถามได้ตลอด 24 ชั่วโมง
ระบบค้นหาคำตอบจาก lecture notes, slides, และตำราเรียน
''',

    'case_healthcare.txt': '''กรณีศึกษา: AI ในการแพทย์ไทย

โรงพยาบาลศิริราชได้นำ AI มาใช้ในการวิเคราะห์ภาพถ่ายทางการแพทย์ (Medical Imaging)
เช่น การตรวจจับมะเร็งปอดจากภาพ X-ray ด้วย Deep Learning
ผลการทดสอบพบว่า AI มีความแม่นยำ 95% เทียบกับรังสีแพทย์ที่ 92%

นอกจากนี้ยังมีการใช้ NLP วิเคราะห์เวชระเบียนอิเล็กทรอนิกส์ (EMR)
เพื่อช่วยแพทย์สรุปประวัติผู้ป่วยและแนะนำการรักษาที่เหมาะสม
ลดเวลาการอ่านเวชระเบียนจาก 15 นาทีเหลือ 2 นาทีต่อเคส

ความท้าทาย: ข้อมูลทางการแพทย์ภาษาไทยมีจำนวนจำกัด
ต้องใช้เทคนิค Transfer Learning จากโมเดลภาษาอังกฤษ
และ Fine-tune ด้วยข้อมูลภาษาไทยเพิ่มเติม''',

    'case_banking.txt': '''กรณีศึกษา: AI ในธนาคารและการเงิน

ธนาคารกสิกรไทยได้พัฒนาระบบ Chatbot ชื่อ KBTG
ที่ใช้ Large Language Model (LLM) ร่วมกับ RAG
ในการตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ทำงานโดยการค้นหาข้อมูลจากฐานความรู้ภายใน
ซึ่งประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
จากนั้น LLM จะสร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center ได้ 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
สามารถให้บริการ 24 ชั่วโมง โดยไม่ต้องรอพนักงาน

เทคโนโลยีที่ใช้: Vector Database สำหรับเก็บ Embedding
Hybrid Search ผสม Dense + Sparse เพื่อค้นหาข้อมูลที่ตรงกับคำถาม''',

    'case_banking_duplicate.txt': '''กรณีศึกษา: AI ในธนาคารและการเงิน

ธนาคารกสิกรไทยได้พัฒนาระบบ Chatbot ชื่อ KBTG
ที่ใช้ Large Language Model (LLM) ร่วมกับ RAG
ในการตอบคำถามลูกค้าเกี่ยวกับผลิตภัณฑ์ทางการเงิน

ระบบ RAG ทำงานโดยการค้นหาข้อมูลจากฐานความรู้ภายใน
ซึ่งประกอบด้วยเอกสารผลิตภัณฑ์ เงื่อนไขบริการ และ FAQ
จากนั้น LLM จะสร้างคำตอบที่เป็นธรรมชาติจากข้อมูลที่ค้นพบ

ผลลัพธ์: ลดภาระ Call Center ได้ 40%
ความพึงพอใจลูกค้าเพิ่มขึ้น 25%
สามารถให้บริการ 24 ชั่วโมง โดยไม่ต้องรอพนักงาน

เทคโนโลยีที่ใช้: Vector Database สำหรับเก็บ Embedding
Hybrid Search ผสม Dense + Sparse เพื่อค้นหาข้อมูลที่ตรงกับคำถาม''',

    'case_education.txt': '''กรณีศึกษา: AI ในการศึกษาไทย

มหาวิทยาลัยหลายแห่งในไทยเริ่มนำ AI มาช่วยในการเรียนการสอน
เช่น ระบบ Intelligent Tutoring System ที่ปรับเนื้อหาตามระดับของผู้เรียน

ตัวอย่างที่น่าสนใจคือการใช้ RAG สร้างระบบถาม-ตอบอัตโนมัติ
สำหรับวิชาเรียน โดยนำเอกสารประกอบการสอน Slides และหนังสือ
มา Embed เป็น Vector แล้วเก็บใน Vector Database

เมื่อนักศึกษาถามคำถาม ระบบจะค้นหาเนื้อหาที่เกี่ยวข้อง
แล้วใช้ LLM สร้างคำตอบพร้อมอ้างอิงแหล่งที่มา

ขั้นตอนสำคัญในการสร้างระบบนี้:
1. เตรียมข้อมูล: แปลง PDF เป็น Markdown
2. ตัดข้อความ: ใช้ Chunking แบ่งเนื้อหาเป็นส่วนย่อย
3. สร้าง Embedding: แปลง Chunk เป็น Vector ด้วยโมเดลที่รองรับภาษาไทย
4. จัดเก็บ: Upsert Vector เข้า Qdrant หรือ Vector DB อื่น
5. ค้นหา: ใช้ Hybrid Search ค้นหา Chunk ที่เกี่ยวข้อง
6. สร้างคำตอบ: ส่ง Context ให้ LLM สร้างคำตอบ

ผลลัพธ์: นักศึกษาสามารถเรียนรู้ด้วยตนเองได้ตลอด 24 ชั่วโมง
ลดภาระอาจารย์ในการตอบคำถามซ้ำๆ ได้กว่า 60%''',

    'case_agriculture.txt': '''กรณีศึกษา: AI ในการเกษตรอัจฉริยะ

Smart Farming ในประเทศไทยใช้ AI วิเคราะห์ภาพถ่ายดาวเทียม
และข้อมูลจาก IoT Sensor เพื่อพยากรณ์ผลผลิตและเฝ้าระวังโรคพืช

ระบบ Computer Vision สามารถตรวจจับโรคข้าวได้จากภาพถ่ายใบข้าว
โดยใช้ Convolutional Neural Network (CNN) ที่ Train ด้วยภาพถ่าย
โรคข้าวกว่า 50,000 ภาพ ครอบคลุม 8 โรคหลัก

นอกจากนี้ยังมีการใช้ NLP วิเคราะห์ข้อมูลราคาสินค้าเกษตร
จากข่าวสาร รายงานภาครัฐ และ Social Media
เพื่อช่วยเกษตรกรตัดสินใจเรื่องการปลูกและการขาย

ความท้าทายของ AI ในการเกษตรไทย:
- ข้อมูลกระจัดกระจายอยู่ในหลายหน่วยงาน
- ภาษาและศัพท์เฉพาะทางการเกษตรไทย
- ต้องทำ Data Engineering เพื่อรวมและทำความสะอาดข้อมูลก่อน'''
}

for fname, content in sample_texts.items():
    with open(f'data/{fname}', 'w', encoding='utf-8') as f:
        f.write(content)

print(f'✅ สร้างไฟล์ตัวอย่าง {len(sample_texts)} ไฟล์ ในโฟลเดอร์ data/')
print()
for fname in sorted(sample_texts.keys()):
    print(f'  📄 {fname}')